## FindMedianEmbeddings_ForEachOfNClusters
This script:
- Reads the expanded dataframe (one row per image) with cluster assignments (output of script 5)
- For a chosen k, computes per-LSOA summary statistics:
    - Count and percentage of images in each cluster
    - Mean, max, and median embedding within each cluster
    - Overall mean, max, and median embedding (across all clusters)
- Saves a single pickle file combining all of the above

In [1]:
import os
import numpy as np
import pandas as pd
from functools import reduce

In [2]:
from directory_filepaths import *
from clustering_functions import global_k
print(f"Number of clusters (defined in clustering_functions.py) = {global_k}")
k = global_k

Number of clusters (defined in clustering_functions.py) = 6


### Get data

In [3]:
# Load expanded dataframe with cluster assignments (output of script 5)
expanded_gdf = pd.read_pickle(os.path.join(data_dir, "one_row_per_image_cleaned_with_cluster_numbers.pkl"))
print(f"Loaded {len(expanded_gdf)} image rows")

Loaded 75586 image rows


# Create a dataframe with % of images in each category, in each LSOA 

In [4]:
category_column = f"scene_cluster_{k}"

# Count images per (LSOA, cluster)
category_counts = (
    expanded_gdf.groupby(["LSOA21CD", category_column])
    .size()
    .reset_index(name="count")
)

# Total images per LSOA
total_counts = expanded_gdf.groupby("LSOA21CD").size().reset_index(name="total_images")

# Merge and compute percentages
category_counts = category_counts.merge(total_counts, on="LSOA21CD")
category_counts["pct"] = category_counts["count"] / category_counts["total_images"] * 100

# Pivot to wide format: one column per cluster for counts and percentages
counts_wide = (
    category_counts.pivot(index="LSOA21CD", columns=category_column, values="count")
    .fillna(0)
    .add_prefix("count_")
)
pct_wide = (
    category_counts.pivot(index="LSOA21CD", columns=category_column, values="pct")
    .fillna(0)
    .add_prefix("pct_")
)

# Combine counts, percentages, and totals per LSOA
lsoa_summary = total_counts.set_index("LSOA21CD").join([counts_wide, pct_wide])

# Sanity check - do the counts sum to total_images?
# Compute sums into local variables instead of creating temporary DataFrame columns
counts_sum = lsoa_summary.filter(like="count_").sum(axis=1)
assert np.all(counts_sum == lsoa_summary["total_images"]), "Counts do not sum to total_images for some LSOAs"
# And do the percentages sum to 100%?
pct_sum = lsoa_summary.filter(like="pct_").sum(axis=1)
assert np.allclose(pct_sum, 100), "Percentages do not sum to 100% for some LSOAs"

lsoa_summary.head()

,total_images,count_1,count_2,count_3,count_4,count_5,count_6,pct_1,pct_2,pct_3,pct_4,pct_5,pct_6
LSOA21CD,,,,,,,,,,,,,
E01004766,64,5.0,10.0,5.0,13.0,19.0,12.0,7.812500,15.625,7.812500,20.312500,29.687500,18.750000
E01004767,72,11.0,9.0,6.0,13.0,16.0,17.0,15.277778,12.500,8.333333,18.055556,22.222222,23.611111
E01004768,44,4.0,0.0,13.0,17.0,9.0,1.0,9.090909,0.000,29.545455,38.636364,20.454545,2.272727
E01004769,40,8.0,1.0,10.0,10.0,9.0,2.0,20.000000,2.500,25.000000,25.000000,22.500000,5.000000
E01004770,40,5.0,1.0,7.0,12.0,12.0,3.0,12.500000,2.500,17.500000,30.000000,30.000000,7.500000


# Find mean/median/max embedding in each LSOA, also by category

In [5]:
def mean_embed(series):
    return np.mean(np.stack(series.values), axis=0)

def max_embed(series):
    return np.max(np.stack(series.values), axis=0)

def median_embed(series):
    return np.median(np.stack(series.values), axis=0)

agg_funcs = {"mean": mean_embed, "max": max_embed, "median": median_embed}
categories = sorted(expanded_gdf[category_column].unique())

all_dfs = []

for agg_name, func in agg_funcs.items():
    dfs = []

    # Per-cluster embedding aggregation
    for cat in categories:
        emb_cat = (
            expanded_gdf[expanded_gdf[category_column] == cat]
            .groupby("LSOA21CD")["embedding"]
            .apply(func)
            .reset_index()
            .rename(columns={"embedding": f"{cat}_{agg_name}"})
        )
        dfs.append(emb_cat)

    # Merge all clusters, then add overall (all images in LSOA)
    merged = reduce(lambda left, right: pd.merge(left, right, on="LSOA21CD", how="outer"), dfs)

    overall = (
        expanded_gdf.groupby("LSOA21CD")["embedding"]
        .apply(func)
        .reset_index()
        .rename(columns={"embedding": f"overall_{agg_name}"})
    )
    merged = merged.merge(overall, on="LSOA21CD", how="left")
    all_dfs.append(merged)

# Combine mean, max, and median into a single DataFrame
final_df = reduce(lambda left, right: pd.merge(left, right, on="LSOA21CD", how="outer"), all_dfs)
print(f"{len(final_df)} LSOAs, {len(final_df.columns) - 1} feature columns")

1695 LSOAs, 21 feature columns


In [6]:
final_df

,LSOA21CD,1_mean,2_mean,3_mean,4_mean,5_mean,6_mean,overall_mean,1_max,2_max,...,5_max,6_max,overall_max,1_median,2_median,3_median,4_median,5_median,6_median,overall_median
0,E01004766,"[0.036080934, -0.07312622, -0.00237751, -0.006...","[0.021999205, -0.047516633, -0.0034092902, -0....","[0.031188201, -0.057373047, -0.014732361, -0.0...","[0.02721735, -0.06162203, 0.014265501, -0.0084...","[0.046590704, -0.063405894, -0.0066067544, -0....","[0.036125183, -0.051254272, 0.013942321, -0.00...","[0.03482639, -0.058570504, 0.0016810745, -0.00...","[0.05166626, -0.04244995, 0.01902771, 0.004512...","[0.049957275, -0.014503479, 0.009437561, 0.014...",...,"[0.06555176, -0.036071777, 0.027816772, 0.0227...","[0.06390381, 0.02444458, 0.02973938, 0.0209045...","[0.06878662, 0.02444458, 0.04058838, 0.0227813...","[0.034698486, -0.08276367, -0.004055023, -0.01...","[0.023117065, -0.050079346, 0.0006117821, -0.0...","[0.032989502, -0.055511475, -0.018936157, 4.17...","[0.024520874, -0.06121826, 0.017410278, -0.011...","[0.04977417, -0.06210327, -0.006793976, -0.006...","[0.035812378, -0.055999756, 0.014129639, -0.00...","[0.037399292, -0.060531616, 0.0037984848, -0.0..."
1,E01004767,"[0.015111879, -0.068403766, -0.008438024, 0.00...","[0.03617944, -0.052797634, -0.011108822, -0.01...","[0.028606415, -0.06138611, -0.023513794, 0.010...","[0.02771407, -0.06703773, 0.010262196, -0.0053...","[0.03915453, -0.0667572, -0.0063030124, -0.005...","[0.03619183, -0.04146733, 0.012488758, -0.0110...","[0.031465285, -0.058895655, -0.0012362666, -0....","[0.04421997, -0.03491211, 0.007724762, 0.04144...","[0.061798096, 0.0012311935, 0.020095825, 0.005...",...,"[0.07910156, -0.040283203, 0.022415161, 0.0113...","[0.0647583, 0.007045746, 0.043884277, 0.013275...","[0.07910156, 0.007045746, 0.057800293, 0.04144...","[0.0088272095, -0.06335449, -0.0055007935, -0....","[0.03152466, -0.059570312, -0.009399414, -0.00...","[0.024368286, -0.049316406, -0.021812439, 0.00...","[0.024017334, -0.057556152, 0.0115737915, -0.0...","[0.036247253, -0.060058594, -0.009237289, -0.0...","[0.037322998, -0.03878784, 0.010101318, -0.013...","[0.028137207, -0.05706787, -0.002474308, -0.00..."
2,E01004768,"[0.035276413, -0.06841278, -0.0051088333, 0.00...",NaN,"[0.052099373, -0.061589167, -0.014418822, 0.00...","[0.03410205, -0.06293263, 0.01156375, -0.01685...","[0.038911607, -0.06541273, -0.0028678046, -0.0...","[0.010543823, -0.05117798, 0.022583008, -0.016...","[0.03997456, -0.06327403, -0.0003300797, -0.00...","[0.06744385, -0.029022217, 0.009536743, 0.0340...",NaN,...,"[0.059020996, -0.04296875, 0.02218628, 0.00774...","[0.010543823, -0.05117798, 0.022583008, -0.016...","[0.08831787, -0.029022217, 0.035186768, 0.0340...","[0.030960083, -0.07192993, -0.008420944, 0.003...",NaN,"[0.048187256, -0.055847168, -0.017730713, 0.00...","[0.033966064, -0.06744385, 0.012634277, -0.019...","[0.036499023, -0.06323242, -0.0031776428, -0.0...","[0.010543823, -0.05117798, 0.022583008, -0.016...","[0.036331177, -0.06199646, -0.0030584335, -0.0..."
3,E01004769,"[0.03569603, -0.058353424, -0.015224934, -0.00...","[0.044647217, -0.0690918, -0.009788513, 0.0082...","[0.048132323, -0.049468994, -0.02563095, -0.00...","[0.034719847, -0.066775516, 0.006770325, -0.01...","[0.029336717, -0.06405979, -0.01206186, -0.007...","[0.061553955, -0.059631348, 0.012565613, -0.01...","[0.038646888, -0.059854127, -0.0100904945, -0....","[0.045013428, -0.041870117, 4.9591064e-05, 0.0...","[0.044647217, -0.0690918, -0.009788513, 0.0082...",...,"[0.057281494, -0.046417236, 0.0036315918, 0.00...","[0.07623291, -0.04095459, 0.016403198, -0.0130...","[0.07623291, -0.026062012, 0.024093628, 0.0186...","[0.035491943, -0.060821533, -0.013225555, -0.0...","[0.044647217, -0.0690918, -0.009788513, 0.0082...","[0.047332764, -0.048706055, -0.02584076, -0.00...","[0.031532288, -0.07077026, 0.010746002, -0.014...","[0.026931763, -0.06726074, -0.010467529, -0.00...","[0.061553955, -0.059631348, 0.012565613, -0.01...","[0.03

### Save

In [7]:
# Attach the LSOA count/percentage summary and save
final_df = final_df.merge(lsoa_summary, on="LSOA21CD")

output_path = os.path.join(data_dir, "per_lsoa_embedding_summaries", "median_embedding_per_cluster.pkl")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
final_df.to_pickle(output_path)
print(f"Saved to {output_path}")

Saved to ../data/processed/per_lsoa_embedding_summaries/median_embedding_per_cluster.pkl


In [8]:
print(final_df['count_1'].sum())
print(final_df['count_2'].sum())

7374.0
11861.0
